In [10]:
import sympy
from sympy import symbols, Matrix, cos, sin, diff, Function
from sympy.physics.mechanics import dynamicsymbols, Lagrangian, Particle, ReferenceFrame, Point
from sympy.physics.mechanics import KanesMethod, inertia, RigidBody
from sympy.abc import t # Time variable

# Deriving Equations of Motion using SymPy

In [11]:
# Define symbolic variables
M_w, M_r, m_b, R, I_w, I_r, d, h, I_b, r, g, T = symbols('M_w M_r m_b R I_w I_r d h I_b r g T')

# Define time-dependent symbolic functions
theta = dynamicsymbols('theta')
x_w = dynamicsymbols('x_w')
x = dynamicsymbols('x')

In [12]:
# Position (adapted from MATLAB script)
# The MATLAB script uses a rotation matrix, let's replicate that logic.
# The original MATLAB code: pos = [cos(theta), sin(theta); -sin(theta), cos(theta)] * [x; h + r] + [x_w; R];

rotation_matrix = Matrix([[cos(theta), sin(theta)], [-sin(theta), cos(theta)]])
body_coords = Matrix([[x], [h + r]])

rotated_body_coords = rotation_matrix * body_coords

x_b = rotated_body_coords[0] + x_w
y_b = rotated_body_coords[1] + R

print("x_b:", sympy.simplify(x_b))
print("y_b:", sympy.simplify(y_b))

x_b: (h + r)*sin(theta(t)) + x(t)*cos(theta(t)) + x_w(t)
y_b: R + (h + r)*cos(theta(t)) - x(t)*sin(theta(t))


In [13]:
# Potential Energy
U_b = m_b * g * y_b
U_r = M_r * g * (R + d * cos(theta))
U_w = M_w * g * R
U = sympy.simplify(U_b + U_r + U_w)

print("Total Potential Energy (U):")
print(U)

Total Potential Energy (U):
g*(M_r*(R + d*cos(theta(t))) + M_w*R + m_b*(R + (h + r)*cos(theta(t)) - x(t)*sin(theta(t))))


In [14]:
# Kinetic Energy
T_w = sympy.Rational(1, 2) * M_w * x_w.diff(t)**2 + sympy.Rational(1, 2) * I_w * (x_w.diff(t)/R)**2
T_r = sympy.Rational(1, 2) * (M_r * x_w.diff(t)**2 + I_r * theta.diff(t)**2)

# T_b - This is the most complex part, carefully translating the MATLAB expression
# MATLAB: 0.5*( m_b*( (diff(x, t)*cos(theta) - x*sin(theta)*diff(theta, t) + (R+r)*cos(theta) + diff(x_w,t) )^2 ...
# + ( -diff(x, t)*sin(theta) - x*cos(theta)*diff(theta, t) - (h+r)*sin(theta) )^2 ) + I_b*(diff(x, t)/r)^2 );

term1_Tb = (x.diff(t)*cos(theta) - x*sin(theta)*theta.diff(t) + (R+r)*cos(theta) + x_w.diff(t))**2
term2_Tb = (x.diff(t)*sin(theta) + x*cos(theta)*theta.diff(t) + (h+r)*sin(theta))**2

T_b = sympy.Rational(1, 2) * (m_b * (term1_Tb + term2_Tb) + I_b * (x.diff(t)/r)**2)

T = sympy.simplify(T_w + T_r + T_b)

print("Total Kinetic Energy (T):")
print(T)

Total Kinetic Energy (T):
(I_b*R**2*Derivative(x(t), t)**2 + I_w*r**2*Derivative(x_w(t), t)**2 + R**2*r**2*(I_r*Derivative(theta(t), t)**2 + M_r*Derivative(x_w(t), t)**2 + M_w*Derivative(x_w(t), t)**2 + m_b*(((h + r)*sin(theta(t)) + x(t)*cos(theta(t))*Derivative(theta(t), t) + sin(theta(t))*Derivative(x(t), t))**2 + ((R + r)*cos(theta(t)) - x(t)*sin(theta(t))*Derivative(theta(t), t) + cos(theta(t))*Derivative(x(t), t) + Derivative(x_w(t), t))**2)))/(2*R**2*r**2)


In [15]:
# Lagrangian
L = T - U
L = sympy.simplify(L)

print("Lagrangian (L):")
print(L)

Lagrangian (L):
(I_b*R**2*Derivative(x(t), t)**2 + I_w*r**2*Derivative(x_w(t), t)**2 - 2*R**2*g*r**2*(M_r*(R + d*cos(theta(t))) + M_w*R + m_b*(R + (h + r)*cos(theta(t)) - x(t)*sin(theta(t)))) + R**2*r**2*(I_r*Derivative(theta(t), t)**2 + M_r*Derivative(x_w(t), t)**2 + M_w*Derivative(x_w(t), t)**2 + m_b*(((h + r)*sin(theta(t)) + x(t)*cos(theta(t))*Derivative(theta(t), t) + sin(theta(t))*Derivative(x(t), t))**2 + ((R + r)*cos(theta(t)) - x(t)*sin(theta(t))*Derivative(theta(t), t) + cos(theta(t))*Derivative(x(t), t) + Derivative(x_w(t), t))**2)))/(2*R**2*r**2)


In [16]:
# Generalized coordinates
q = [x_w, theta, x]

# Equations of Motion (Euler-Lagrange)
eom = []
for qi in q:
    eq = diff(diff(L, qi.diff(t)), t) - diff(L, qi)
    eom.append(sympy.simplify(eq))

print("Equations of Motion:")
for i, eq in enumerate(eom):
    print(f"Equation for {q[i]}:")
    # print(eq)
    print(eq)
    print("\n" + "-"*50 + "\n")

Equations of Motion:
Equation for x_w(t):
(I_w*Derivative(x_w(t), (t, 2)) + R**2*(M_r*Derivative(x_w(t), (t, 2)) + M_w*Derivative(x_w(t), (t, 2)) - m_b*((R + r)*sin(theta(t))*Derivative(theta(t), t) + x(t)*sin(theta(t))*Derivative(theta(t), (t, 2)) + x(t)*cos(theta(t))*Derivative(theta(t), t)**2 + 2*sin(theta(t))*Derivative(theta(t), t)*Derivative(x(t), t) - cos(theta(t))*Derivative(x(t), (t, 2)) - Derivative(x_w(t), (t, 2)))))/R**2

--------------------------------------------------

Equation for theta(t):
I_r*Derivative(theta(t), (t, 2)) - M_r*d*g*sin(theta(t)) + R**2*m_b*sin(2*theta(t))/2 + R*m_b*r*sin(2*theta(t)) + R*m_b*sin(theta(t))*Derivative(x_w(t), t) + R*m_b*sin(2*theta(t))*Derivative(x(t), t)/2 - g*h*m_b*sin(theta(t)) - g*m_b*r*sin(theta(t)) - g*m_b*x(t)*cos(theta(t)) - h**2*m_b*sin(2*theta(t))/2 - h*m_b*r*sin(2*theta(t)) - h*m_b*sin(2*theta(t))*Derivative(x(t), t)/2 + m_b*r*sin(theta(t))*Derivative(x_w(t), t) + m_b*x(t)**2*Derivative(theta(t), (t, 2)) - m_b*x(t)*sin(theta(t